# Processes
Run CPU-heavy or isolated work in another process.


In [ ]:
# A separate process isolates CPU-heavy work from the API event loop.
import asyncio
import sys

process = await asyncio.create_subprocess_exec(
    sys.executable, "-c", "print(sum(i*i for i in range(10)))",
    stdout=asyncio.subprocess.PIPE,
)
output, _ = await process.communicate()
print(int(output))


## Polished version
Encapsulate process creation, validate input, and check the exit status.


In [ ]:
# Encapsulate process creation, validation, and result parsing in one adapter.
class CpuProcess:
    async def sum_squares(self, limit: int) -> int:
        if limit < 0:
            raise ValueError("limit must be non-negative")
        code = "import sys; n=int(sys.argv[1]); print(sum(i*i for i in range(n)))"
        # Pass arguments directly; avoiding a shell prevents command injection.
        child = await asyncio.create_subprocess_exec(
            sys.executable, "-c", code, str(limit), stdout=asyncio.subprocess.PIPE
        )
        stdout, _ = await child.communicate()
        if child.returncode != 0:
            raise RuntimeError("worker process failed")
        return int(stdout)

print(await CpuProcess().sum_squares(10))
